# Solar Filament V28 — HighRes-Expert / Novel-Instance Recovery

**Competition-grade T4 notebook.** A genuinely new trained *solar-pretrained*, high-resolution instance-segmentation detector. It fine-tunes the verified V20 YOLO11m-seg weights on independent 1024-pixel crops, sampling all official MAGFiLO expert versions without mixing annotators within a crop. The model is evaluated on whether it finds **instances V25 missed**, not whether it outputs lots of novel masks.

- **Control:** exact original V25 Kaggle `0.39` CSV (`ac470fbfa0e2ca5f2d...`) always preserved. New model never replaces it merely because it trained successfully.
- **T4 training:** 1024px crops (no 768-to-128 reduction), pretrained V20 solar backbone, one-class polygon-instance supervision, with optional context-scale model.
- **Inference:** overlapping 1024px native tiles and 1280px contextual windows. Full COCO RLE masks, conservative cross-window deduplication and optional evidence-based mask stitching.
- **Quality:** train a held-day, out-of-fold *novel-object* selector; positive training examples must match an expert instance **not matched by V25**. Ablate raw, ranker-gated and conservative fusion.
- **Evaluation:** per-annotator pooled PQ, 53-scene calibration/51-scene audit, paired date bootstrap, TP/FP/FN and unique-recall diagnostics. The reused historical split is **not an untouched holdout**.
- **Reproducibility:** checkpoint files saved before test inference, trainer/data manifest, output ZIPs and historical CSV SHA verification. No external test annotations.

**Inputs:** the official `solar3k` MAGFiLO dataset and saved V25 trained models/CSV archive (or files directly), as used in your successful V27 run. Enable Kaggle **T4 GPU** and Internet for dependency installation. **Run All.** Default trains one strong high-resolution branch; enable context branch if GPU allowance permits. Public 0.40 is an objective, **not a guaranteed score**.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); OUT=Path('/kaggle/working/solar_v28')
SEED=2026; GPU_ID=0; H=W=2048
RAW_CONF=.08; RAW_IOU=.72; MAX_DET=120
ALLOW_NON_V20_WEIGHT_HASH=False; REQUIRE_V24_MODELS=True; FULL_VALIDATION_REQUIRED=True
V25_EXPECTED_QUALITY_CUTOFF=.38
V25_EXPECTED_CSV_SHA='ac470fbfa0e2ca5f2d';V25_EXPECTED_ROWS=1093
CROP_SIDE=320
CURVE_WINDOW=768;CURVE_STRIDE=640;CURVE_SIDE=512;CURVE_MIN_AREA=170;CURVE_MAX_PROPOSALS=42
# V28 dedicated new pretrained detector; GPU time controlled here.
TRAIN_HR=True; TRAIN_CONTEXT=False; REUSE_TRAINED_IF_FOUND=True
TRAIN_CROPS_PER_SCENE=3; VALID_CROPS_PER_SCENE=2
HR_CROP=1024; HR_EPOCHS=18; HR_BATCH=2; HR_IMGSZ=1024
CONTEXT_CROP=1280; CONTEXT_EPOCHS=13; CONTEXT_BATCH=3; CONTEXT_IMGSZ=1024
TRAIN_INNER_HOLDOUT_FRACTION=.10; HR_LEARNING_RATE=0.00016
# A threshold here governs detector output only; PQ selects the calibrated gate.
NEW_RAW_CONF=.11; NEW_RAW_IOU=.66; NEW_MAX_DET=90
MIN_NEW_AREA=160; MAX_NEW_PER_SCENE=75
STITCH_OVERLAP_FRAC=.64; STITCH_IOU=.36
FUSION_MIN_FREE=.70; FUSION_NMS_IOU=.24
QUALITY_THRESHOLDS=[.25,.35,.45,.55,.65,.75]
MIN_LOCAL_GAIN_TO_PROMOTE=.004
ALWAYS_EXPORT_ABLATIONS=True
for sub in ('models','raw','refined','curve_raw','reports','submissions','input_data','hr_dataset','new_raw','train_runs','gate_cache'):
    (OUT/sub).mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(),'Attach Kaggle solar3k and your V25 trained models.'
print('V28 root:',OUT)

In [ ]:
import os,sys,time,json,hashlib,zipfile,shutil,gzip,random,subprocess,importlib.util
import gc as _gc_safe
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

def require(module,package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
require('pycocotools','pycocotools')
# Lock exact V22 version when available to maximize baseline reproduction.
try:
    from importlib.metadata import version as package_version
    installed_version=package_version('ultralytics')
except Exception:
    installed_version=None
if installed_version!='8.4.189':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.189'])
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4/T4x2. No CPU or nano fallback.'
assert torch.cuda.device_count()>GPU_ID
print('Using',torch.cuda.get_device_name(GPU_ID),'ultralytics',ultralytics.__version__)
cv2.setNumThreads(0);random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
H=W=2048
EXT={'.jpeg','.jpg','.png','.tif','.tiff','.bmp'}
def files_in(d):
    p=Path(d)
    return sorted(q for q in p.iterdir() if q.is_file() and q.suffix.lower() in EXT) if p.is_dir() else []
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp');temp.write_text(json.dumps(obj,indent=2,default=str));temp.replace(p)
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def txt(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def as_rle(x):return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
assert callable(_gc_safe.collect)
save_json(OUT/'reports'/'env.json',{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(GPU_ID),'no_prior_split_required':True,'prior_verified_public_PQ':0.39,'target_public_PQ':0.40})

## 1. Official data and the historical V20 acquisition-day partition

In [ ]:
ANAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found=sorted(INPUT.rglob(ANAME))
OFFICIAL=next((p.parent.parent for p in found if (p.parent.parent/'test'/'test_images').is_dir() and (p.parent.parent/'train'/'train_images').is_dir()),None)

def find_zip(name):
    matches=list(INPUT.rglob(name))
    return matches[0] if matches else None

def safe_images_unzip(archive,destination):
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not item.filename.lower().endswith(tuple(EXT)):continue
            target=destination/Path(item.filename).name
            if not target.exists():
                with z.open(item) as src,open(target,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    return destination
if OFFICIAL:
    ANNOT=OFFICIAL/'train'/ANAME
    TRAIN={p.stem:p for p in files_in(OFFICIAL/'train'/'train_images')}
    TEST={p.stem:p for p in files_in(OFFICIAL/'test'/'test_images')}
    print('Using full official competition folder',OFFICIAL)
else:
    annot=found[0] if found else None
    ztest=find_zip('test.zip');ztrain=find_zip('sometrain.zip')
    assert annot is not None and ztest is not None, 'Attach official solar3k, or the annotation JSON and test.zip as Kaggle inputs.'
    ANNOT=annot
    TEST={p.stem:p for p in files_in(safe_images_unzip(ztest,OUT/'input_data'/'test_images'))}
    TRAIN={p.stem:p for p in files_in(safe_images_unzip(ztrain,OUT/'input_data'/'train_images'))} if ztrain else {}
    print('Using upload ZIP fallback. Validation may cover fewer than 104 physical images.')
assert len(TEST)==180,('Expected 180 test scenes; found',len(TEST))
assert len(set(TEST)&set(TRAIN))==0
DOC=json.loads(ANNOT.read_text())
BY_ID=defaultdict(list)
for ann in DOC['annotations']:BY_ID[str(ann['image_id'])].append(ann)
VERSIONS=defaultdict(list)
for item in DOC['images']:
    stem=Path(item['file_name']).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item['id']),BY_ID.get(str(item['id']),[])))
assert set(TRAIN)==set(VERSIONS)
if TRAIN:
    days=defaultdict(list)
    for img in DOC['images']:
        stem=Path(img['file_name']).stem
        days[stem[:8]].append(stem)
    days={k:sorted(set(v)) for k,v in days.items()}
    shuffled=sorted(days)
    random.Random(SEED+2020).shuffle(shuffled)
    selected_days=[];n=0
    for d in shuffled:
        if n>=104:break
        selected_days.append(d);n+=len(days[d])
    VALIDATION=sorted(set(stem for d in selected_days for stem in days[d]) & set(TRAIN))
else:VALIDATION=[]
FULL_V20_VALIDATION=(len(VALIDATION)==104 and len(TRAIN)==707)
print('Images: train present',len(TRAIN),'test',len(TEST),'V20 tune available',len(VALIDATION),'full reproduction',FULL_V20_VALIDATION)
if VALIDATION:
    assert not ({s[:8] for s in VALIDATION} & {s[:8] for s in TRAIN if s not in VALIDATION})
save_json(OUT/'reports'/'dataset.json',{'annotation_sha256':sha(ANNOT),'train_available':len(TRAIN),
    'test_images':len(TEST),'validation_available':len(VALIDATION),'full_v20_reproduction':FULL_V20_VALIDATION,
    'validation_stems':VALIDATION,'test_ground_truth_used':False})

## 2. Restore verified frozen V20 detectors

In [ ]:
EXPECT={
 'portable_global_m':('portable_global_m_best.pt','73fd3f3a4931'),
 'portable_native_m':('portable_native_m_best.pt','1a932d28e590'),
}
V20_DIR=Path('/kaggle/input/datasets/lighittha/solarouts/solar_v20_from_scratch/models')
ZIPS=sorted(set(p for p in INPUT.rglob('*.zip') if any(token in p.name.lower() for token in ('v20','v21','weights','models'))))
WEIGHTS={};HASHES={};SOURCES={}
(OUT/'models').mkdir(parents=True,exist_ok=True)
for model_name,(filename,prefix) in EXPECT.items():
    path=OUT/'models'/filename
    choices=[]
    direct=V20_DIR/filename
    if direct.is_file():choices.append(direct)
    choices+=[p for p in INPUT.rglob(filename) if p.is_file() and p!=direct]
    if path.is_file():choices.insert(0,path)
    if not path.is_file() or path.stat().st_size<20_000_000:
        restored=False
        for src in choices:
            if 30_000_000<src.stat().st_size<65_000_000:
                if src!=path:shutil.copy2(src,path)
                SOURCES[model_name]=str(src);restored=True;break
        if not restored:
            for archive in ZIPS:
                try:
                    with zipfile.ZipFile(archive) as z:
                        member='models/'+filename
                        if member not in z.namelist():continue
                        if not (30_000_000<z.getinfo(member).file_size<65_000_000):continue
                        temp=path.with_suffix('.part')
                        with z.open(member) as rf,open(temp,'wb') as wf:shutil.copyfileobj(rf,wf,1<<20)
                        temp.replace(path);SOURCES[model_name]=str(archive)+'::'+member
                        restored=True;break
                except (OSError,zipfile.BadZipFile) as e:print('Ignore corrupt ZIP:',archive,e)
        if not restored:raise FileNotFoundError('Missing trained V20 '+filename+f'. Expected at {V20_DIR/filename} or inside the V20 weights ZIP. This notebook never substitutes pretrained-only or nano models.')
    digest=sha(path)
    if not ALLOW_NON_V20_WEIGHT_HASH:
        assert digest.startswith(prefix),f'{filename}: wrong weights hash! Expected V20 prefix {prefix}, got {digest[:12]}'
    m=YOLO(str(path))
    n_params=sum(p.numel() for p in m.model.parameters())
    assert m.task=='segment' and 21_000_000<=n_params<=24_000_000,(model_name,m.task,n_params)
    del m;_gc_safe.collect()
    WEIGHTS[model_name]=path;HASHES[model_name]=digest
print('Recovered V20 model identity:',{k:v[:12] for k,v in HASHES.items()})
save_json(OUT/'reports'/'weights.json',{'sha256':HASHES,'source':SOURCES,'V20_checkpoint_hash_gate':not ALLOW_NON_V20_WEIGHT_HASH})

## 3. Immutable V22 detector proposal generation and original mask selection

In [ ]:
# Locked V22 successful source definitions.
SRC_BASE_GLOBAL=('portable_global_m','g1536')
SRC_BASE_NATIVE=('portable_native_m','g1536')
SRC_HI=('portable_global_m','g1792')
SRC_TILES=('portable_native_m','t1280')
SRC_HIFLIP=('portable_global_m','g1792_hflip')
BASE_SOURCES=[SRC_HI,SRC_TILES]
EXTRA_SOURCES=[]
ACTIVE=BASE_SOURCES+EXTRA_SOURCES
ENGINES={}
def get_engine(name):
    if name not in ENGINES:
        ENGINES.clear();_gc_safe.collect();torch.cuda.empty_cache()
        ENGINES[name]=YOLO(str(WEIGHTS[name]))
    return ENGINES[name]

def view_windows(view):
    if view=='g1536':return [(0,0,2048,1536,False)]
    if view=='g1792':return [(0,0,2048,1792,False)]
    if view=='g1792_hflip':return [(0,0,2048,1792,True)]
    if view=='t1280':return [(x,y,1280,1024,False) for y in (0,768) for x in (0,768)]
    raise ValueError(view)
def sig(source):
    name,view=source
    return hashlib.sha256(json.dumps({'model_sha256':HASHES[name],'view':view,
      'conf':RAW_CONF,'iou':RAW_IOU,'max_det':MAX_DET,'retina_masks':True,
      'agnostic_nms':True,'ultralytics':ultralytics.__version__,'version':'V22_1' if view in ('g1792','t1280','g1536') else 'V24_1'},sort_keys=True).encode()).hexdigest()
def cache_file(source,stem):
    name,view=source
    return OUT/'raw'/name/view/(stem+'.json.gz')
@torch.inference_mode()
def raw_predictions(source,stem):
    path=cache_file(source,stem);identifier=sig(source)
    if path.exists():
        try:
            with gzip.open(path,'rt') as f:v=json.load(f)
            if v.get('signature')==identifier and v.get('stem')==stem:return v['predictions']
        except (OSError,ValueError,KeyError):pass
    image_path=TRAIN.get(stem) or TEST.get(stem)
    if image_path is None:raise FileNotFoundError('Missing scene: '+stem)
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W),(stem, None if gray is None else gray.shape)
    predictions=[]
    name,view=source
    model=get_engine(name)
    for x,y,side,net,hflip in view_windows(view):
        roi=np.ascontiguousarray(gray[y:y+side,x:x+side])
        if hflip:roi=cv2.flip(roi,1)
        image=cv2.cvtColor(roi,cv2.COLOR_GRAY2BGR)
        result=model.predict(image,imgsz=net,conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,
                             agnostic_nms=True,retina_masks=True,device=GPU_ID,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mask_data=result.masks.data.detach().cpu().numpy()
        confidences=result.boxes.conf.detach().cpu().numpy()
        for mask_arr,conf in zip(mask_data,confidences):
            mask=(mask_arr>0.5).astype(np.uint8)
            if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
            if hflip:mask=np.ascontiguousarray(cv2.flip(mask,1))
            if int(mask.sum())<100:continue
            if side!=2048:
                ys,xs=np.where(mask)
                truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10) or
                               (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=mask
            else:truncated=False;full=mask
            rr=enc(full)
            predictions.append({'counts':txt(rr),'score':float(conf),'area':int(coco.area(rr)),
                                'truncated':truncated,'source_view':view})
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with gzip.open(tmp,'wt',compresslevel=3) as f:
        json.dump({'signature':identifier,'stem':stem,'view':view,'predictions':predictions},f,separators=(',',':'))
    tmp.replace(path)
    return predictions
print('V23 views including immutable winner:',ACTIVE)

In [ ]:
def recipe(name,sources,gconf=.24,tconf=.32,area=200,seam=False,refine='none',minfree=.45):
    return {'name':name,'sources':[list(s) for s in sources],'global_conf':float(gconf),
            'tile_conf':float(tconf),'min_area':int(area),'seam_guard':bool(seam),
            'nms_iou':.50,'tile_bonus':.08,'min_free':float(minfree),'refine':refine}

# This exact configuration produced V22_ALT_2.csv (public PQ 0.37).
WINNER=recipe('V22_ALT2_WINNER_037',[SRC_HI,SRC_TILES],
              gconf=.24,tconf=.42,area=200,seam=True,refine='none',minfree=.45)
WINNER['nms_iou']=.50
WINNER['tile_bonus']=.08
WINNER['engine']='v22'
WINNER['fusion']='none'
WINNER['tile_policy']='baseline'

def experiment(name,sources=None,gconf=.24,tconf=.42,area=200,seam=True,
               bonus=.08,nms=.50,minfree=.45,tile_policy='baseline',fusion='none'):
    r=recipe(name,sources if sources is not None else [SRC_HI,SRC_TILES],
             gconf=gconf,tconf=tconf,area=area,seam=seam,minfree=minfree)
    r.update(engine='v23',tile_bonus=bonus,nms_iou=nms,tile_policy=tile_policy,fusion=fusion)
    return r


def modify_rle(rr,operation):
    if operation=='none':return rr
    box=coco.toBbox(rr);x,y,w,h=[int(q) for q in box]
    if w<1 or h<1:return rr
    pad=3;x0=max(0,x-pad);y0=max(0,y-pad);x1=min(W,x+w+pad);y1=min(H,y+h+pad)
    full=coco.decode(rr).astype(np.uint8)
    roi=full[y0:y1,x0:x1]
    if operation=='close3':
        result=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    elif operation=='dilate1':
        result=cv2.dilate(roi,np.ones((3,3),np.uint8),iterations=1)
    elif operation=='erode1':
        result=cv2.erode(roi,np.ones((3,3),np.uint8),iterations=1)
    else:raise ValueError(operation)
    if int(result.sum())<50:return rr
    full[y0:y1,x0:x1]=result
    return enc(full)

def select_masks(raw,cfg):
    sources={tuple(pair) for pair in cfg['sources']}
    candidates=[]
    for source,items in raw.items():
        if source not in sources:continue
        threshold=cfg['tile_conf'] if source[1].startswith('t') else cfg['global_conf']
        for p in items:
            score=float(p['score']);area=int(p['area'])
            if score<threshold or area<cfg['min_area']:continue
            if cfg['seam_guard'] and source[1].startswith('t') and p.get('truncated',False):continue
            rank=score+(cfg['tile_bonus'] if source[1].startswith('t') else 0.)
            candidates.append((rank,area,p))
    candidates.sort(key=lambda q:(-q[0],-q[1]))
    masks=[];originals=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in candidates:
        rr=as_rle(p['counts'])
        if originals:
            ov=np.asarray(coco.iou([rr],originals,[0]*len(originals)))[0]
            if np.any(ov>=cfg['nms_iou']):continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        free_area=int(free.sum())
        if free_area<cfg['min_area'] or free_area/max(1,area)<cfg['min_free']:continue
        cleaned=enc(free)
        if cfg.get('refine','none')!='none':cleaned=modify_rle(cleaned,cfg['refine'])
        # Retain exact V20 occupied behavior for baseline; for refined recipes use actual refined footprint.
        if cfg.get('refine','none')!='none':
            shaped=coco.decode(cleaned).astype(np.uint8)&(occupied==0)
            if int(shaped.sum())<cfg['min_area']:continue
            cleaned=enc(shaped)
            occupied|=shaped
        else:occupied|=free
        masks.append(cleaned);originals.append(rr)
        if len(masks)>=120:break
    return masks

# Do not change the winner branch; these tests protect the baseline.
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['global_conf']==.24 and WINNER['tile_conf']==.42
assert WINNER['seam_guard'] and WINNER['min_area']==200
assert WINNER['nms_iou']==.5 and WINNER['tile_bonus']==.08
assert callable(_gc_safe.collect)

## 4. Local expert-version PQ proxy (same definition as V25)

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            obj=coco.frPyObjects(seg,H,W)
            return coco.merge(obj) if isinstance(obj,list) else obj
        rr=dict(seg)
        if isinstance(rr.get('counts'),str):rr['counts']=rr['counts'].encode('ascii')
        return rr
    if len(seg)>0 and isinstance(seg[0],(int,float)):seg=[seg]
    polygons=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not polygons:return None
    rr=coco.frPyObjects(polygons,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
TRUTH={}
def gt(stem):
    if stem not in TRUTH:
        TRUTH[stem]=[[rr for a in arr if (rr:=annotation_rle(a)) is not None]
                     for _,arr in VERSIONS[stem]]
    return TRUTH[stem]
def match_counts(truth,preds):
    if not truth:return np.array([0.,0.,float(len(preds)),0.])
    if not preds:return np.array([0.,0.,0.,float(len(truth))])
    ious=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    # Match the earlier V20 PQ proxy EXACTLY for control-score parity.
    ii,jj=linear_sum_assignment(-ious)
    ok=ious[ii,jj]>.5
    tp=int(ok.sum())
    return np.array([float(ious[ii[ok],jj[ok]].sum()),float(tp),
                     float(len(preds)-tp),float(len(truth)-tp)])
def physical_counts(stem,preds):
    result=np.zeros(4,float)
    for version in gt(stem):result+=match_counts(version,preds)
    return result
def pq(count):
    den=count[1]+.5*(count[2]+count[3])
    return float(count[0]/den) if den>0 else 0.
def metrics(count):
    den=count[1]+.5*(count[2]+count[3])
    return {'PQ':pq(count),'SQ':float(count[0]/count[1]) if count[1] else 0.,
       'RQ':float(count[1]/den) if den else 0.,'TP':int(count[1]),
       'FP':int(count[2]),'FN':int(count[3])}
example=np.zeros((H,W),np.uint8);example[40:61,50:92]=1
assert coco.area(as_rle(txt(enc(example))))==example.sum()
assert abs(pq(match_counts([enc(example)],[enc(example)]))-1)<1e-9
print('PQ scorer and full-size RLE roundtrip: PASS')

## 5. Recover trained V24 and V25 model components; preserve the real 0.39 pipeline

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib, math, time, io
from torchvision.models import resnet18

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class PromptUNet(nn.Module):
    def __init__(self):
        super().__init__();e=resnet18(weights=None);old=e.conv1
        e.conv1=nn.Conv2d(4,64,7,2,3,bias=False)
        self.stem=nn.Sequential(e.conv1,e.bn1,e.relu);self.pool=e.maxpool
        self.e1=e.layer1;self.e2=e.layer2;self.e3=e.layer3;self.e4=e.layer4
        self.d4=DoubleConv(768,192);self.d3=DoubleConv(320,128)
        self.d2=DoubleConv(192,96);self.d1=DoubleConv(160,64)
        self.out=nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                               nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),nn.Conv2d(32,1,1))
    def forward(self,x):
        e0=self.stem(x);e1=self.e1(self.pool(e0));e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        def up(a,b):return torch.cat([F.interpolate(a,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1)
        a=self.d4(up(e4,e3));a=self.d3(up(a,e2));a=self.d2(up(a,e1));a=self.d1(up(a,e0))
        return self.out(a)

def make_channels(gray_uint8,mask_uint8):
    gray=np.asarray(gray_uint8,dtype=np.uint8)
    equal=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    blur=cv2.GaussianBlur(gray,(3,3),0)
    dx=cv2.Sobel(blur,cv2.CV_32F,1,0,ksize=3);dy=cv2.Sobel(blur,cv2.CV_32F,0,1,ksize=3)
    edge=np.clip(cv2.magnitude(dx,dy)/240.,0,1)
    return np.stack([gray/255.,equal/255.,mask_uint8.astype(np.float32),edge],axis=0).astype(np.float32)

def normalized_bbox_roi(box,scale=1.48,min_extent=120):
    x,y,w,h=map(float,box)
    if w<=0 or h<=0:return None
    dx=min(W,max(float(min_extent),w*scale));dy=min(H,max(float(min_extent),h*scale))
    cx=x+w/2;cy=y+h/2
    x0=int(np.clip(round(cx-dx/2),0,W-1));y0=int(np.clip(round(cy-dy/2),0,H-1))
    x1=int(np.clip(round(cx+dx/2),x0+1,W));y1=int(np.clip(round(cy+dy/2),y0+1,H))
    return x0,y0,x1,y1

def find_original_asset(filename):
    direct=sorted(INPUT.rglob(filename))
    for p in direct:
        if p.is_file() and p.stat().st_size>100:return p.read_bytes(),str(p)
    for z in sorted(INPUT.rglob('*.zip')):
        if not any(token in z.name.lower() for token in ('v24','model','trained')):continue
        try:
            with zipfile.ZipFile(z) as arc:
                matches=[m for m in arc.namelist() if Path(m).name==filename]
                if matches:return arc.read(matches[0]),str(z)+'::'+matches[0]
                # V24 full output bundles a second ZIP containing its scored CSVs.
                for nested in [m for m in arc.namelist() if m.lower().endswith('.zip')]:
                    try:
                        with zipfile.ZipFile(io.BytesIO(arc.read(nested))) as inner:
                            hits=[x for x in inner.namelist() if Path(x).name==filename]
                            if hits:return inner.read(hits[0]),str(z)+'::'+nested+'::'+hits[0]
                    except (OSError,zipfile.BadZipFile):pass
        except (OSError,zipfile.BadZipFile):continue
    return None,None

for filename in ('SOLAR_V24_REFINER_BEST.pt','SOLAR_V24_INSTANCE_QUALITY.joblib'):
    target=OUT/'models'/filename
    data,source=find_original_asset(filename)
    if data is None:
        if REQUIRE_V24_MODELS:raise FileNotFoundError('Attach V24 model archive; missing '+filename)
    else:
        if not target.exists() or sha(target)!=hashlib.sha256(data).hexdigest():target.write_bytes(data)
        print('Recovered V24 asset:',filename, 'from',source,'SHA:',sha(target)[:16])

V24_REFINER_PATH=OUT/'models'/'SOLAR_V24_REFINER_BEST.pt'
V24_QUALITY_PATH=OUT/'models'/'SOLAR_V24_INSTANCE_QUALITY.joblib'
refiner_v24=PromptUNet().to('cuda:'+str(GPU_ID))
refiner_v24.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v24.eval()
meta_v24=joblib.load(V24_QUALITY_PATH)
quality_v24=meta_v24['model'];V24_FEATURES=meta_v24['features']
assert len(V24_FEATURES)==15 and callable(quality_v24.predict_proba)
print('Recovered genuine V24 quality model, OOF AUROC:',meta_v24.get('oof_auc'))

In [ ]:
def matching_scores(rr,records):
    if not records:return 0.,0.,0.
    rles=[as_rle(q['counts']) for q in records]
    ious=np.asarray(coco.iou([rr],rles,[0]*len(rles)),float).reshape(-1)
    i=int(ious.argmax());return float(ious[i]),float(records[i]['score']),float(max(q['score'] for q in records))

@torch.inference_mode()
def run_refiner(model,gray,rr,threshold=.50,min_prompt_iou=.54):
    crop=normalized_bbox_roi([int(v) for v in coco.toBbox(rr)],scale=1.48)
    if crop is None:return rr
    x0,y0,x1,y1=crop;original=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    if not original.any():return rr
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(original,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(make_channels(image,prompt))[None].cuda(GPU_ID)
    with torch.autocast('cuda',dtype=torch.float16):
        probabilities=model(inp).sigmoid()[0,0].float().cpu().numpy()
    refined=(cv2.resize(probabilities,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=threshold).astype(np.uint8)
    oa=int(original.sum());na=int(refined.sum())
    if na<80 or not (.52<=na/max(oa,1)<=1.65):return rr
    overlap=int((refined&original).sum())/max(1,oa+na-int((refined&original).sum()))
    if overlap<min_prompt_iou:return rr
    full=np.zeros((H,W),np.uint8);full[y0:y1,x0:x1]=refined
    return enc(full)

def load_base(stem):
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    return raw,select_masks(raw,WINNER)

def v24_refined_masks(stem,baseline,gray=None):
    key=sha(V24_REFINER_PATH)[:18];p=OUT/'refined'/('v24_'+key+'_'+stem+'.json.gz')
    if p.is_file():
        with gzip.open(p,'rt') as f:v=json.load(f)
        if len(v)==len(baseline):return [as_rle(x) for x in v]
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    proposed=[run_refiner(refiner_v24,gray,rr) for rr in baseline]
    occupied=np.zeros((H,W),np.uint8);final=[]
    for a,b in zip(baseline,proposed):
        f=coco.decode(b).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:f=coco.decode(a).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:continue
        final.append(enc(f));occupied|=f
    if len(final)!=len(baseline):final=baseline
    p.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(p,'wt',compresslevel=3) as f:json.dump([txt(v) for v in final],f)
    return final

def features_v24(stem,base,refined,gray=None,raw=None):
    raw=raw if raw is not None else {SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    rows=[]
    for rr,new in zip(base,refined):
        x,y,w,h=map(float,coco.toBbox(rr));mask=coco.decode(rr).astype(np.uint8);area=int(mask.sum())
        dilation=cv2.dilate(mask,np.ones((11,11),np.uint8))
        ring=(dilation>0)&(mask==0)
        inside=float(gray[mask>0].mean()) if area else 0.;outside=float(gray[ring].mean()) if ring.any() else inside
        ga,gscore,gmax=matching_scores(rr,raw[SRC_HI]);ta,tscore,tmax=matching_scores(rr,raw[SRC_TILES])
        na=float(coco.area(new));oa=float(coco.area(rr))
        riou=float(coco.iou([rr],[new],[0])[0,0])
        rows.append({'base_score':max(gscore,tscore),'base_area_log':float(np.log1p(area)),
                     'base_box_ar':float(np.log((w+1)/(h+1))),'base_fill':float(area/max(1,w*h)),
                     'base_center_x':float((x+w/2)/W),'base_center_y':float((y+h/2)/H),
                     'global_agreement':ga,'tile_agreement':ta,'global_maxconf':gmax,'tile_maxconf':tmax,
                     'pixel_avg_inside':inside/255.,'pixel_avg_ring':outside/255.,
                     'contrast':(outside-inside)/255.,'refined_iou':riou,'refined_area_ratio':na/max(oa,1.)})
    return pd.DataFrame(rows,columns=V24_FEATURES)

def v24_control(stem):
    raw,base=load_base(stem)
    if not base:return []
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=v24_refined_masks(stem,base,gray)
    df=features_v24(stem,base,refined,gray,raw)
    qs=quality_v24.predict_proba(df[V24_FEATURES].fillna(0.))[:,1]
    return [rr for rr,q in zip(base,qs) if q>=.25]  # Exact V24 QUALITY_025: no extra clipping

s=np.zeros((H,W),np.uint8);s[50:90,70:130]=1
assert int(coco.area(enc(s)))==2400
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['tile_conf']==.42 and WINNER['global_conf']==.24 and WINNER['seam_guard']
print('V24 control frozen. New V25 scoring cannot alter its thresholds or masks.')

In [ ]:
# Restore all genuinely trained V25 checkpoint bytes; no substitute/pretrained-only networks.
from skimage.morphology import skeletonize
from skimage.segmentation import watershed

V25_REAL_PATH=OUT/'models'/'SOLAR_V25_REAL_REFINER_BEST.pt'
V25_CURVE_PATH=OUT/'models'/'SOLAR_V25_CURVE_BEST.pt'
V25_QUALITY_PATH=OUT/'models'/'SOLAR_V25_EXTENDED_QUALITY.joblib'
for p in (V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH):
    data,source=find_original_asset(p.name)
    if data is None:raise FileNotFoundError('Missing '+p.name+'. Attach the complete V25 trained checkpoint ZIP.')
    if not p.exists() or sha(p)!=hashlib.sha256(data).hexdigest():p.write_bytes(data)
    print('Recovered frozen V25',p.name,sha(p)[:18], 'from',source)
refiner_real=PromptUNet().cuda(GPU_ID)
refiner_real.load_state_dict(torch.load(V25_REAL_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_real.eval();REAL_REFINER_SHA=sha(V25_REAL_PATH)

def real_refine(stem,rr,gray=None):
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    return run_refiner(refiner_real,gray,rr,threshold=.5,min_prompt_iou=.50)

v25_ranker_package=joblib.load(V25_QUALITY_PATH)
new_rankers=v25_ranker_package['models']
V25_QUALITY_CUTOFF=float(v25_ranker_package['cutoff'])
print('Frozen V25 ranker cutoff:',V25_QUALITY_CUTOFF,'feature count:',len(v25_ranker_package['columns']))
assert abs(V25_QUALITY_CUTOFF-V25_EXPECTED_QUALITY_CUTOFF)<1e-6, 'Wrong V25 ranking checkpoint/cutoff'

def rank_probs(models,df):
    if df.empty:return np.asarray([],float)
    a=df[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
    return np.mean([m.predict_proba(a)[:,1] for m in models],axis=0)


In [ ]:
class MiniUNet(nn.Module):
    def __init__(self,channels=2,outputs=3):
        super().__init__()
        self.a=DoubleConv(channels,24);self.b=DoubleConv(24,48);self.c=DoubleConv(48,96)
        self.d=DoubleConv(96,160)
        self.up3=DoubleConv(160+96,96);self.up2=DoubleConv(96+48,64)
        self.up1=DoubleConv(64+24,32);self.final=nn.Conv2d(32,outputs,1)
    def forward(self,x):
        a=self.a(x);b=self.b(F.avg_pool2d(a,2));c=self.c(F.avg_pool2d(b,2));d=self.d(F.avg_pool2d(c,2))
        u=self.up3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.up2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.up1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.final(u)


curve_path=V25_CURVE_PATH
curve_net=MiniUNet().cuda(GPU_ID)
curve_net.load_state_dict(torch.load(curve_path,map_location='cpu',weights_only=False)['model'],strict=True)
curve_net.eval();CURVE_SHA=sha(curve_path)
print('Loaded frozen V25 curve network',CURVE_SHA[:18])

In [ ]:
def curve_locations():
    starts=list(range(0,H-CURVE_WINDOW+1,CURVE_STRIDE))
    if starts[-1]!=H-CURVE_WINDOW:starts.append(H-CURVE_WINDOW)
    return [(x,y) for y in starts for x in starts]
CURVE_LOCATIONS=curve_locations()
assert len(CURVE_LOCATIONS)==9, CURVE_LOCATIONS

def curve_input(gray):
    equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(gray)
    return torch.from_numpy(np.stack([gray/255.,equ/255.],0).astype(np.float32))[None].cuda(GPU_ID)

@torch.inference_mode()
def curve_proposals(stem):
    cache=OUT/'curve_raw'/(stem+'.json.gz')
    signature=hashlib.sha256((CURVE_SHA+stem+str(CURVE_WINDOW)+str(CURVE_SIDE)).encode()).hexdigest()
    if cache.exists():
        try:
            with gzip.open(cache,'rt') as f:v=json.load(f)
            if v.get('identity')==signature:return v['proposals']
        except (ValueError,OSError,KeyError):pass
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(stem)
    props=[]
    for x0,y0 in CURVE_LOCATIONS:
        raw=image[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        inp=cv2.resize(raw,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        with torch.autocast('cuda',dtype=torch.float16):
            probs=curve_net(curve_input(inp)).sigmoid()[0].float().cpu().numpy()
        fg=cv2.resize(probs[0],(CURVE_WINDOW,CURVE_WINDOW))
        center=cv2.resize(probs[1],(CURVE_WINDOW,CURVE_WINDOW))
        seg=(fg>.53).astype(np.uint8)
        n,components,stats,_=cv2.connectedComponentsWithStats(seg,connectivity=8)
        for k in range(1,n):
            component_area=int(stats[k,cv2.CC_STAT_AREA])
            if component_area<CURVE_MIN_AREA:continue
            local=(components==k).astype(np.uint8)
            # Conservatively split only when two *distinct* centerline islands exist.
            line=(center>.56).astype(np.uint8)&local
            m,markers,mark_stats,_=cv2.connectedComponentsWithStats(line,connectivity=8)
            good=[q for q in range(1,m) if mark_stats[q,cv2.CC_STAT_AREA]>=8]
            pieces=[local]
            if 2<=len(good)<=5 and component_area>700:
                seeds=np.zeros_like(local,np.int32)
                for label_idx,mark_idx in enumerate(good,1):seeds[markers==mark_idx]=label_idx
                regions=watershed(-cv2.distanceTransform(local,cv2.DIST_L2,3),seeds,mask=(local>0))
                candidates=[(regions==z).astype(np.uint8) for z in range(1,len(good)+1)]
                if all(int(c.sum())>=CURVE_MIN_AREA for c in candidates):pieces=candidates
            for part in pieces:
                area=int(part.sum())
                if area<CURVE_MIN_AREA:continue
                sy,sx=np.where(part)
                if min(sx)<8 and x0>0 or max(sx)>CURVE_WINDOW-9 and x0+CURVE_WINDOW<W:continue
                if min(sy)<8 and y0>0 or max(sy)>CURVE_WINDOW-9 and y0+CURVE_WINDOW<H:continue
                local_gray=raw[part>0];mask_score=float(np.mean(fg[part>0]))
                line_score=float(np.mean(center[part>0]))
                if mask_score<.59 or line_score<.18:continue
                # Relatively dark features, not arbitrary bright solar active-region structures.
                patch=raw[max(0,sy.min()-16):min(CURVE_WINDOW,sy.max()+17),
                          max(0,sx.min()-16):min(CURVE_WINDOW,sx.max()+17)]
                if len(local_gray) and float(np.median(local_gray))>float(np.percentile(patch,75))+8:continue
                full=np.zeros((H,W),np.uint8)
                full[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]=part
                rr=enc(full)
                props.append({'counts':txt(rr),'score':float(np.clip(.65*mask_score+.35*line_score,0,1)),
                              'area':int(coco.area(rr)),'truncated':False,'source_view':'curve_v25',
                              'center_support':line_score})
    props.sort(key=lambda p:p['score'],reverse=True)
    distinct=[];rles=[]
    for p in props:
        rr=as_rle(p['counts'])
        if rles and float(np.max(coco.iou([rr],rles,[0]*len(rles))))>.33:continue
        distinct.append(p);rles.append(rr)
        if len(distinct)>=CURVE_MAX_PROPOSALS:break
    with gzip.open(cache,'wt',compresslevel=3) as f:
        json.dump({'identity':signature,'stem':stem,'proposals':distinct},f,separators=(',',':'))
    return distinct

def extend_with_curve(stem,baseline,raw=None):
    # Existing instance identities are never silently split or replaced here.
    raw=raw if raw is not None else load_base(stem)[0]
    base_rles=list(baseline)
    base_all=base_rles.copy()
    occupied=np.zeros((H,W),np.uint8)
    for rr in base_rles:occupied|=coco.decode(rr).astype(np.uint8)
    additions=[]
    for p in curve_proposals(stem):
        rr=as_rle(p['counts'])
        if base_all and float(np.max(coco.iou([rr],base_all,[0]*len(base_all))))>=.25:continue
        part=coco.decode(rr).astype(np.uint8)
        free=part&(occupied==0)
        if int(free.sum())<CURVE_MIN_AREA or free.sum()/max(1,part.sum())<.70:continue
        accepted=enc(free)
        additions.append(accepted);occupied|=free;base_all.append(accepted)
    return base_rles+additions,['detector']*len(base_rles)+['curve']*len(additions)

print('Curve candidate generation initialized:',len(CURVE_LOCATIONS),'windows per 2048 scene')

In [ ]:
from scipy.spatial.distance import cdist

# These 40+ new features are computed from *predicted masks/images only*.
EXTRA_FIELDS=['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
 'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count','skeleton_length',
 'skeleton_area_ratio','mean_thickness','max_thickness','width_variation','local_component_count',
 'intensity_inside_std','intensity_inside_p10','intensity_inside_p50','intensity_inside_p90',
 'intensity_ring_p50','intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
 'sobel_ring_mean','contrast_normalized','sun_radial','rim_distance','mask_near_solar_limb',
 'hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos',
 'neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
 'curve_origin','curve_conf','curve_center_support','curve_vs_global_iou','curve_vs_tile_iou',
 'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
 'mask_global_rank','mask_tile_rank','mask_area_over_disk','intensity_outside_std']
NEW_FEATURES=list(V24_FEATURES)+EXTRA_FIELDS
assert len(NEW_FEATURES)>60

def image_entropy(pixels):
    if len(pixels)==0:return 0.
    hist=np.bincount((pixels//16).astype(np.uint8),minlength=16).astype(float)
    hist=hist/max(hist.sum(),1);h=hist[hist>0]
    return float(-(h*np.log(h)).sum())

def extract_extra(gray,rr,all_boxes,source_kind,curve_meta,raw):
    x,y,w,h=map(float,coco.toBbox(rr));x0=max(0,int(x)-12);y0=max(0,int(y)-12)
    x1=min(W,int(np.ceil(x+w))+12);y1=min(H,int(np.ceil(y+h))+12)
    # COCO decode is full-frame but geometric operations are limited to a small crop.
    mask=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    g=gray[y0:y1,x0:x1];area=int(mask.sum());yy,xx=np.where(mask>0)
    ring=cv2.dilate(mask,np.ones((9,9),np.uint8))&(1-mask)
    inside=g[mask>0];outside=g[ring>0]
    if not len(inside):inside=np.asarray([0],np.uint8)
    if not len(outside):outside=inside
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
    hull_area=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
    # Downsample skeleton for speed; normalize pixel-to-image ratio.
    ratio=min(1.,150/max(mask.shape));sw=max(8,int(mask.shape[1]*ratio));sh=max(8,int(mask.shape[0]*ratio))
    reduced=(cv2.resize(mask,(sw,sh),interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
    sk=skeletonize(reduced>0).astype(np.uint8)
    neigh=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk.astype(np.int16)
    endpoints=int(((sk>0)&(neigh==1)).sum());branches=int(((sk>0)&(neigh>=3)).sum())
    dist=cv2.distanceTransform(reduced,cv2.DIST_L2,3)[reduced>0]
    components=cv2.connectedComponents(reduced,connectivity=8)[0]-1
    if len(xx)>1:
        cov=np.cov(np.stack([xx,yy]).astype(float));ev,vec=np.linalg.eigh(cov)
        major=vec[:,np.argmax(ev)];elong=float(np.sqrt((max(ev)+1)/(min(ev)+1)))
        ecc=float(np.sqrt(max(0.,1-min(ev)/(max(ev)+1e-6))))
    else:major=np.array([1.,0.]);elong=1.;ecc=0.
    sob=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
    hu=cv2.HuMoments(cv2.moments(mask)).flatten()
    cx=x+w/2;cy=y+h/2;rad=np.hypot(cx-W/2,cy-H/2)/(W/2)
    other_boxes=np.asarray(all_boxes,float).reshape(-1,4)
    current_centroid=np.array([cx,cy]);centroids=other_boxes[:,:2]+other_boxes[:,2:]/2 if len(other_boxes) else np.zeros((0,2))
    distances=np.linalg.norm(centroids-current_centroid,axis=1) if len(centroids) else np.asarray([])
    nearby=sorted(float(z) for z in distances if z>1)
    bbox_ious=[]
    for bx,by,bw,bh in other_boxes:
        if abs(bx-x)+abs(by-y)+abs(bw-w)+abs(bh-h)<.001:continue
        iw=max(0,min(x+w,bx+bw)-max(x,bx));ih=max(0,min(y+h,by+bh)-max(y,by))
        bbox_ious.append(iw*ih/max(1,w*h+bw*bh-iw*ih))
    raw_g=raw[SRC_HI];raw_t=raw[SRC_TILES]
    gi,gc,gmax=matching_scores(rr,raw_g);ti,tc,tmax=matching_scores(rr,raw_t)
    flat=dict(log_perimeter=np.log1p(perimeter),log_convex_area=np.log1p(hull_area),
        solidity=area/max(1,hull_area),extent=area/max(1,w*h),elongation=min(100,elong),
        eccentricity=ecc,aspect_absolute=max((w+1)/(h+1),(h+1)/(w+1)),area_frac_box=area/max(1,w*h),
        edge_density=perimeter/max(1,area),endpoint_count=endpoints,branchpoint_count=branches,
        skeleton_length=int(sk.sum())/max(.15,ratio),skeleton_area_ratio=float(sk.sum())/max(1,reduced.sum()),
        mean_thickness=float(np.mean(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        max_thickness=float(np.max(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        width_variation=float(np.std(dist)) if len(dist) else 0.,local_component_count=max(0,components),
        intensity_inside_std=float(inside.std())/255.,intensity_inside_p10=float(np.percentile(inside,10))/255.,
        intensity_inside_p50=float(np.median(inside))/255.,intensity_inside_p90=float(np.percentile(inside,90))/255.,
        intensity_ring_p50=float(np.median(outside))/255.,
        intensity_diff_p10=float(np.percentile(outside,10)-np.percentile(inside,10))/255.,
        intensity_diff_mean=float(outside.mean()-inside.mean())/255.,inside_entropy=image_entropy(inside),
        sobel_inside_mean=float(sob[mask>0].mean())/255. if area else 0.,
        sobel_ring_mean=float(sob[ring>0].mean())/255. if ring.any() else 0.,
        contrast_normalized=float(outside.mean()-inside.mean())/max(10.,float(outside.std())),
        sun_radial=rad,rim_distance=max(0,1-rad),mask_near_solar_limb=float(rad>.93),
        hull_vs_mask=hull_area/max(1,area),compactness=perimeter**2/max(1,4*np.pi*area),
        hu_1=float(np.sign(hu[0])*np.log1p(abs(hu[0]))),hu_2=float(np.sign(hu[1])*np.log1p(abs(hu[1]))),
        pca_angle_sin=float(major[1]),pca_angle_cos=float(major[0]),
        neighbor_center_dist=nearby[0]/2048 if nearby else 1.,
        neighbor_bbox_iou=max(bbox_ious,default=0.),
        neighbor_area_ratio=float(sorted([abs(np.log((w*h+1)/(bw*bh+1))) for bx,by,bw,bh in other_boxes])[1]) if len(other_boxes)>1 else 0.,
        n_neighbors_100=sum(v<100 for v in nearby),curve_origin=float(source_kind=='curve'),
        curve_conf=float(curve_meta.get('score',0)),curve_center_support=float(curve_meta.get('center_support',0)),
        curve_vs_global_iou=gi,curve_vs_tile_iou=ti,
        raw_global_count=len(raw_g),raw_tile_count=len(raw_t),
        raw_high_conf_global_count=sum(p['score']>.4 for p in raw_g),
        raw_high_conf_tile_count=sum(p['score']>.4 for p in raw_t),
        mask_global_rank=gc,mask_tile_rank=tc,mask_area_over_disk=area/(np.pi*1024**2),
        intensity_outside_std=float(outside.std())/255.)
    assert set(flat)==set(EXTRA_FIELDS),('Missing feature',set(EXTRA_FIELDS)-set(flat))
    return flat

def extended_features(stem,masks,origins=None,curve_meta=None,gray=None,raw=None):
    if not masks:return pd.DataFrame(columns=NEW_FEATURES)
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    if raw is None:raw=load_base(stem)[0]
    # Refinement-based V24 features need the frozen, correct V24 refiner.
    if origins is None:origins=['detector']*len(masks)
    if curve_meta is None:curve_meta=[{} for _ in masks]
    assert len(masks)==len(origins)==len(curve_meta)
    refined=v24_refined_masks(stem,masks,gray) if all(o=='detector' for o in origins) else [
        run_refiner(refiner_v24,gray,r) if o=='detector' else r for r,o in zip(masks,origins)]
    base=features_v24(stem,masks,refined,gray,raw)
    boxes=[list(map(float,coco.toBbox(rr))) for rr in masks]
    for i,rr in enumerate(masks):
        extra=extract_extra(gray,rr,boxes,origins[i],curve_meta[i],raw)
        for k in EXTRA_FIELDS:base.loc[i,k]=extra[k]
    return base[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)

print('Extended feature count:',len(NEW_FEATURES))

In [ ]:
VAL_STEMS=sorted(VALIDATION)
assert len(VAL_STEMS)==104 or not FULL_VALIDATION_REQUIRED
by_day=defaultdict(list)
for st in VAL_STEMS:by_day[st[:8]].append(st)
all_days=sorted(by_day);random.Random(SEED+952).shuffle(all_days)
CAL_DAYS=set(all_days[:int(len(all_days)*.52)])
CAL=sorted(s for s in VAL_STEMS if s[:8] in CAL_DAYS)
AUDIT=sorted(s for s in VAL_STEMS if s[:8] not in CAL_DAYS)
assert len(CAL)>30 and len(AUDIT)>30 and not ({s[:8] for s in CAL}&{s[:8] for s in AUDIT})
print('Train quality on',len(CAL),'scenes, evaluate on',len(AUDIT),'different-day scenes')

# Cache masks and features separately from the checkpointed models.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:
        full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    meta=[]
    for index,rr in enumerate(full):
        if origins[index]=='detector':meta.append({})
        else:
            index_ious=np.asarray(coco.iou([rr],[as_rle(v['counts']) for v in curve_list],
                                             [0]*len(curve_list)),float).reshape(-1) if curve_list else np.asarray([])
            meta.append(curve_list[int(index_ious.argmax())] if len(index_ious) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    path=OUT/'reports'/('features_'+stem+'.joblib')
    # Data are caches of MODEL PREDICTIONS, not ground-truth test labels.
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    obj={'signature':signature,'masks':[txt(x) for x in masks], 'origins':origins,
         'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def consensus_quality_labels(stem,preds):
    expert_versions=gt(stem);out=[]
    for rr in preds:
        votes=[]
        for truth in expert_versions:
            if truth:
                vec=np.asarray(coco.iou([rr],truth,[0]*len(truth)),float).reshape(-1)
                votes.append(float(np.max(vec)) if len(vec) else 0.)
            else:votes.append(0.)
        out.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return out


assert list(v25_ranker_package['columns'])==NEW_FEATURES,'V25 feature order mismatch'


In [ ]:
# Use all frozen V25 components to reconstruct its exact candidate-order/quality behavior.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    curve_rles=[as_rle(p['counts']) for p in curve_list]
    meta=[]
    for i,rr in enumerate(full):
        if origins[i]=='detector':meta.append({});continue
        overlaps=np.asarray(coco.iou([rr],curve_rles,[0]*len(curve_rles)),float).reshape(-1) if curve_rles else np.asarray([])
        meta.append(curve_list[int(overlaps.argmax())] if len(overlaps) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    location=OUT/'reports'/('v25_features_'+stem+'.joblib')
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+sha(V25_QUALITY_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if location.is_file():
        try:
            obj=joblib.load(location)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    result={'signature':signature,'masks':[txt(r) for r in masks], 'origins':origins,
            'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(result,location,compress=3)
    return result

def nonoverlap_after_refinement(masks):
    occupied=np.zeros((H,W),np.uint8);out=[]
    for rr in masks:
        pix=coco.decode(rr).astype(np.uint8)&(occupied==0)
        if int(pix.sum())<80:continue
        out.append(enc(pix));occupied|=pix
    return out

def frozen_v25_masks(stem, obj=None):
    info=obj if obj is not None else scene_feature_data(stem)
    prob=rank_probs(new_rankers,info['features'])
    inds=[i for i in range(info['n_base']) if prob[i]>=V25_QUALITY_CUTOFF]
    masks=[as_rle(info['masks'][i]) for i in inds]
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=[real_refine(stem,rr,gray) if prob[i]>=.50 else rr for i,rr in zip(inds,masks)]
    return nonoverlap_after_refinement(refined)

assert V25_QUALITY_CUTOFF==V25_EXPECTED_QUALITY_CUTOFF
assert list(v25_ranker_package['columns'])==NEW_FEATURES
print('V25 0.39 candidate logic restored, original ranker/curve/refiner frozen.')


## 6. Prepare genuinely instance-labeled *native-resolution* expert crops

YOLO polygons follow the documented `<class> x1 y1 ...` normalized format. Each crop uses **one consistent expert annotation version**, retaining separate filament instances. The crop builder rasterizes each instance independently, clips it to the window and converts each preserved component to a simplified polygon. Crops are split by acquisition day; all 104 historical detector-validation scenes remain outside new-detector training. We record how many disconnected components are simplified.

**Compute note:** the original V20 backbone has seen the 603 training scenes. Consequently the inner validation split is new-branch-separated but not pretraining-independent. The outer 104-scene split remains the more relevant development audit (and has been repeatedly reused).

In [ ]:
import math, csv
import yaml
from skimage.morphology import skeletonize
HR_ROOT=OUT/'hr_dataset'
ANNOT_FINGERPRINT=sha(ANNOT)
SPLITS={'train':[],'val':[]}
TRAIN_STEMS=sorted(set(TRAIN)-set(VALIDATION))
assert len(TRAIN_STEMS)==603 and len(VALIDATION)==104
by_train_day=defaultdict(list)
for s in TRAIN_STEMS:by_train_day[s[:8]].append(s)
all_train_days=sorted(by_train_day);random.Random(SEED+281).shuffle(all_train_days)
inner_days=set(all_train_days[:max(10,int(len(all_train_days)*TRAIN_INNER_HOLDOUT_FRACTION))])
HR_TRAIN_STEMS=[s for s in TRAIN_STEMS if s[:8] not in inner_days]
HR_VAL_STEMS=[s for s in TRAIN_STEMS if s[:8] in inner_days]
assert HR_TRAIN_STEMS and HR_VAL_STEMS
assert not ({s[:8] for s in HR_TRAIN_STEMS}&{s[:8] for s in HR_VAL_STEMS})
assert not(set(HR_TRAIN_STEMS+HR_VAL_STEMS)&set(VALIDATION))

# Rasterize all polygons of ONE annotation into one instance mask; never merge two instances.
def instance_roi_mask(annotation,x0,y0,side):
    mask=np.zeros((side,side),np.uint8)
    seg=annotation.get('segmentation',[])
    if isinstance(seg,dict):
        rr=annotation_rle(annotation)
        if rr is not None:mask[:]=coco.decode(rr)[y0:y0+side,x0:x0+side].astype(np.uint8)
    else:
        if seg and isinstance(seg[0],(float,int)):seg=[seg]
        for poly in seg:
            if len(poly)<6 or len(poly)%2:continue
            pts=np.asarray(poly,np.float32).reshape(-1,2)
            pts=np.rint(pts-[x0,y0]).astype(np.int32)
            cv2.fillPoly(mask,[pts],1)
    return mask

def convert_instance_to_yolo(mask,side):
    n,components,stats,_=cv2.connectedComponentsWithStats(mask,8)
    useful=[k for k in range(1,n) if stats[k,cv2.CC_STAT_AREA]>=50]
    if not useful:return None,0.0
    # The polygon label format cannot encode multiple disconnected islands of one instance.
    # The dominant island is preserved; this fraction is logged as a potential information loss.
    k=max(useful,key=lambda j:stats[j,cv2.CC_STAT_AREA])
    primary=(components==k).astype(np.uint8)
    contours,_=cv2.findContours(primary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not contours:return None,0.0
    contour=max(contours,key=cv2.contourArea)
    epsilon=max(.5,.0015*cv2.arcLength(contour,True))
    approx=cv2.approxPolyDP(contour,epsilon,True).reshape(-1,2)
    if len(approx)<3:return None,0.0
    if len(approx)>300:approx=approx[np.linspace(0,len(approx)-1,300).astype(int)]
    poly=np.clip(approx.astype(float)/side,0,1)
    if np.unique(poly,axis=0).shape[0]<3:return None,0.0
    return '0 '+' '.join(f'{v:.6f}' for v in poly.flatten()),float(primary.sum())/max(1,int(mask.sum()))

# Validate converter using a REAL official annotation, then compare polygon geometry.
ex_stem=next(s for s in TRAIN_STEMS if VERSIONS[s][0][1])
ex_a=VERSIONS[ex_stem][0][1][0]
raw_mask=instance_roi_mask(ex_a,0,0,H)
assert raw_mask.shape==(H,W) and raw_mask.sum()>0
bx,by,bw,bh=[int(v) for v in ex_a['bbox']]
xx=int(np.clip(bx-25,0,W-HR_CROP));yy=int(np.clip(by-25,0,H-HR_CROP))
lab,coverage=convert_instance_to_yolo(instance_roi_mask(ex_a,xx,yy,HR_CROP),HR_CROP)
assert lab is not None and lab.startswith('0 '),(ex_stem,ex_a['bbox'])
assert all(0<=float(s)<=1 for s in lab.split()[1:])
print('PASS real annotation→YOLO polygon:',len(lab.split()[1:])//2,'vertices, largest-island ratio:',round(coverage,3))

In [ ]:
BANK_CONFIG={'annotation_sha256':ANNOT_FINGERPRINT,'seed':SEED,'native_crop':HR_CROP,
             'crops_per_train_scene':TRAIN_CROPS_PER_SCENE,'crops_per_val_scene':VALID_CROPS_PER_SCENE,
             'train_stems_sha':hashlib.sha256(json.dumps(HR_TRAIN_STEMS).encode()).hexdigest(),
             'val_stems_sha':hashlib.sha256(json.dumps(HR_VAL_STEMS).encode()).hexdigest()}
BANK_HASH=hashlib.sha256(json.dumps(BANK_CONFIG,sort_keys=True).encode()).hexdigest()
MANIFEST=HR_ROOT/'dataset_manifest.json'
if MANIFEST.is_file():
    prior=json.loads(MANIFEST.read_text())
    assert prior['bank_hash']==BANK_HASH,'Existing crop bank has a different version. Clear solar_v28/hr_dataset.'

stats={'images':0,'objects':0,'dropped_objects':0,'source_scenes':set(),'lost_island_fraction':[]}
for split,stems,repeat in [('train',HR_TRAIN_STEMS,TRAIN_CROPS_PER_SCENE),('val',HR_VAL_STEMS,VALID_CROPS_PER_SCENE)]:
    for sub in ('images','labels'):(HR_ROOT/sub/split).mkdir(parents=True,exist_ok=True)
    for stem in tqdm(stems,desc='Build '+split+' expert-crop labels'):
        versions=[arr for _,arr in VERSIONS[stem] if arr]
        if not versions:continue
        gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
        if gray is None:raise FileNotFoundError(stem)
        rng=random.Random(SEED+int(hashlib.sha256((stem+split).encode()).hexdigest()[:8],16))
        for idx in range(repeat):
            name=f'{stem}_crop{idx:02d}'
            im_path=HR_ROOT/'images'/split/(name+'.jpg')
            lab_path=HR_ROOT/'labels'/split/(name+'.txt')
            # Version rotates across crops, preserving annotation diversity and expert integrity.
            expert=versions[(idx+rng.randrange(len(versions)))%len(versions)]
            if idx==repeat-1 and repeat>=3:
                x0=rng.randrange(0,W-HR_CROP+1);y0=rng.randrange(0,H-HR_CROP+1)
            else:
                anchor=rng.choice(expert)
                bx,by,bw,bh=map(float,anchor.get('bbox',[0,0,0,0]))
                cx=bx+bw/2+rng.uniform(-215,215);cy=by+bh/2+rng.uniform(-215,215)
                x0=int(np.clip(round(cx-HR_CROP/2),0,W-HR_CROP));y0=int(np.clip(round(cy-HR_CROP/2),0,H-HR_CROP))
            if im_path.exists() and lab_path.exists():
                continue
            lines=[]
            for a in expert:
                bx,by,bw,bh=map(float,a.get('bbox',[0,0,0,0]))
                if bx+bw<x0 or bx>=x0+HR_CROP or by+bh<y0 or by>=y0+HR_CROP:continue
                cropped=instance_roi_mask(a,x0,y0,HR_CROP)
                line,ratio=convert_instance_to_yolo(cropped,HR_CROP)
                if line:
                    lines.append(line);stats['lost_island_fraction'].append(1-ratio)
                else:stats['dropped_objects']+=1
            im=cv2.cvtColor(gray[y0:y0+HR_CROP,x0:x0+HR_CROP],cv2.COLOR_GRAY2BGR)
            assert im.shape==(HR_CROP,HR_CROP,3)
            tmp_im=im_path.with_suffix('.partial.jpg')
            assert cv2.imwrite(str(tmp_im),im,[cv2.IMWRITE_JPEG_QUALITY,96])
            tmp_im.replace(im_path)
            tmp_lab=lab_path.with_suffix('.partial.txt');tmp_lab.write_text('\n'.join(lines)+('\n' if lines else ''))
            tmp_lab.replace(lab_path)
            stats['images']+=1;stats['objects']+=len(lines);stats['source_scenes'].add(stem)
    SPLITS[split]=sorted((HR_ROOT/'images'/split).glob('*.jpg'))
    assert SPLITS[split],('No crop images in',split)

YAML_PATH=HR_ROOT/'filaments.yaml'
YAML_PATH.write_text(yaml.safe_dump({'path':str(HR_ROOT),'train':'images/train','val':'images/val','names':{0:'filament'}},sort_keys=False))
BANK_META={**BANK_CONFIG,'bank_hash':BANK_HASH,'train_image_count':len(SPLITS['train']),
           'val_image_count':len(SPLITS['val']),'new_crops_built':stats['images'],
           'new_instance_labels':stats['objects'],'dropped_instances':stats['dropped_objects'],
           'max_lost_island_fraction':max(stats['lost_island_fraction'],default=0)}
save_json(MANIFEST,BANK_META)
print('HR labeled crops:',BANK_META['train_image_count'],BANK_META['val_image_count'])
print('New annotations written:',stats['objects'],'max discarded island share:',BANK_META['max_lost_island_fraction'])
# Inspect 10 train labels; every segmentation row must have >=3 coordinate pairs.
for fp in sorted((HR_ROOT/'labels'/'train').glob('*.txt'))[:10]:
    for row in fp.read_text().splitlines():
        terms=row.split();assert terms[0]=='0' and len(terms)>=7 and (len(terms)-1)%2==0,fp

## 7. Train **pretrained** high-resolution YOLO11m-seg

We fine-tune the **actual verified V20 native** segmentation checkpoint (solar-adapted weights), not ImageNet-only weights or random network initialization. Training uses high-resolution 1024px crops with every filament as a separate polygon instance. Mosaic/mixup/copy-paste are disabled to avoid mixing inconsistent experts and breaking long structures; horizontal and vertical flips remain valid. The second contextual branch is optional because two full training runs can exceed a single Kaggle allocation.

The model's `best.pt` checkpoint is copied to `/kaggle/working/SOLAR_V28_HIGHRES_BEST.pt` **before** test inference. Attaching this checkpoint on later Kaggle runs reuses it. Resume of a partially completed Ultralytics run is attempted only when its full `last.pt` trainer checkpoint exists; best-weight-only reattachment skips training.

In [ ]:
# Keep the three FROZEN V25 feature/refiner models off GPU while YOLO fine-tunes.
# Their parameters are unchanged; move them back before frozen V25 inference.
for frozen_model in (refiner_v24,refiner_real,curve_net):frozen_model.cpu()
_gc_safe.collect();torch.cuda.empty_cache()
print('Froze V25 feature networks on CPU for memory-safe high-resolution detector training.')

In [ ]:
HR_ALIAS=Path('/kaggle/working/SOLAR_V28_HIGHRES_BEST.pt')
CTX_ALIAS=Path('/kaggle/working/SOLAR_V28_CONTEXT_BEST.pt')

def get_saved_checkpoint(alias):
    if alias.is_file():return alias
    payload,source=find_original_asset(alias.name)
    if payload is not None:
        alias.write_bytes(payload)
        print('Recovered trained V28 checkpoint',alias.name,'from',source)
    return alias if alias.exists() else None

def train_new_yolo(tag,starting_model,alias,epochs,imgsz,batch,lr,enabled=True):
    checkpoint=get_saved_checkpoint(alias)
    if checkpoint is None:
        if not enabled:raise FileNotFoundError('Model branch disabled and no saved checkpoint: '+str(alias))
        runroot=OUT/'train_runs';name='V28_'+tag
        last=runroot/name/'weights'/'last.pt'
        model=YOLO(str(last if last.is_file() else starting_model))
        train_kwargs=dict(data=str(YAML_PATH),task='segment',epochs=epochs,imgsz=imgsz,batch=batch,
              device=GPU_ID,workers=2,cache=False,amp=True,optimizer='AdamW',lr0=lr,lrf=.12,
              cos_lr=True,patience=9,deterministic=True,seed=SEED+28,save=True,
              save_period=5,plots=False,verbose=False,project=str(runroot),name=name,exist_ok=True,
              mosaic=0.0,mixup=0.0,copy_paste=0.0,degrees=0.0,translate=.05,scale=.12,
              fliplr=.50,flipud=.50,hsv_h=0.0,hsv_s=0.0,hsv_v=.10,
              mask_ratio=2,overlap_mask=True,single_cls=True)
        if last.is_file():
            print('Resume interrupted Ultralytics trainer from:',last)
            # This retains its original train configuration and optimizer state.
            model.train(resume=True,device=GPU_ID)
        else:
            print('Fine-tuning VERIFIED solar-pretrained V20 checkpoint:',starting_model)
            model.train(**train_kwargs)
        best=runroot/name/'weights'/'best.pt'
        assert best.is_file(),'Training did not write best.pt; inspect Kaggle train-run directory.'
        shutil.copy2(best,alias)
    network=YOLO(str(alias))
    assert network.task=='segment' and 21_000_000<sum(p.numel() for p in network.model.parameters())<24_000_000
    print('V28',tag,'checkpoint SHA:',sha(alias)[:20], 'size MiB:',round(alias.stat().st_size/2**20,1))
    return network

model_hr=train_new_yolo('HIGHRES',WEIGHTS['portable_native_m'],HR_ALIAS,
                        HR_EPOCHS,HR_IMGSZ,HR_BATCH,HR_LEARNING_RATE,TRAIN_HR)
if TRAIN_CONTEXT or get_saved_checkpoint(CTX_ALIAS) is not None:
    model_ctx=train_new_yolo('CONTEXT',WEIGHTS['portable_global_m'],CTX_ALIAS,
                            CONTEXT_EPOCHS,CONTEXT_IMGSZ,CONTEXT_BATCH,HR_LEARNING_RATE*.65,TRAIN_CONTEXT)
else:
    model_ctx=None
print('V28 models ready. No test labels accessed.')

## 8. Overlapping native-scale inference and conservative deduplication

The 1024 model sees original 1024 image pixels per tile. A second inference view provides 1280-pixel context downscaled to 1024 input; this view uses the same trained model and therefore does not cost an additional training run. Optional contextual checkpoint can be added as another independent source. Boundary fragments are retained as hypotheses, *not blindly rejected*. We only combine them when masks strongly agree in overlapping native coordinates. Every candidate's original mask and model score is retained for audit.

In [ ]:
# Restore the EXACT frozen V25 models to CUDA after YOLO training.
for frozen_model in (refiner_v24,refiner_real,curve_net):frozen_model.cuda(GPU_ID).eval()
print('Restored verified V25 inference networks; V28 trainer has not modified their weights.')

In [ ]:
DEVICE_ID=GPU_ID
V28_SOURCES=[('hr_1024',model_hr,HR_ALIAS,1024,1024,512),
             ('hr_context',model_hr,HR_ALIAS,1280,1024,768)]
if model_ctx is not None:
    V28_SOURCES.append(('ctx_independent',model_ctx,CTX_ALIAS,1280,1024,768))

def window_starts(side,stride):
    starts=list(range(0,H-side+1,stride))
    if not starts or starts[-1]!=H-side:starts.append(H-side)
    return starts

def rle_best_pairwise_iou(rr,collection):
    return float(np.max(coco.iou([rr],collection,[0]*len(collection)))) if collection else 0.

@torch.inference_mode()
def new_detector_scene(stem):
    params={'sources':[(n,sha(w)) for n,_,w,_,_,_ in V28_SOURCES],
       'conf':NEW_RAW_CONF,'iou':NEW_RAW_IOU,'max_det':NEW_MAX_DET,
       'min_area':MIN_NEW_AREA,'stitch_overlap':STITCH_OVERLAP_FRAC,'stitch_iou':STITCH_IOU}
    signature=hashlib.sha256(json.dumps(params,sort_keys=True).encode()).hexdigest()
    path=OUT/'new_raw'/(stem+'.json.gz')
    if path.is_file():
        try:
            with gzip.open(path,'rt') as f:cached=json.load(f)
            if cached.get('signature')==signature:return cached['items']
        except (OSError,ValueError,KeyError):pass
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W)
    raw=[]
    for source_name,engine,weight,side,net,stride in V28_SOURCES:
        for y0 in window_starts(side,stride):
            for x0 in window_starts(side,stride):
                tile=np.ascontiguousarray(gray[y0:y0+side,x0:x0+side])
                rgb=cv2.cvtColor(tile,cv2.COLOR_GRAY2BGR)
                pred=engine.predict(rgb,imgsz=net,conf=NEW_RAW_CONF,iou=NEW_RAW_IOU,
                  max_det=NEW_MAX_DET,retina_masks=True,agnostic_nms=True,
                  device=DEVICE_ID,verbose=False)[0]
                if pred.masks is None or pred.boxes is None:continue
                masks=pred.masks.data.detach().cpu().numpy()
                scores=pred.boxes.conf.detach().cpu().numpy()
                for mm,score in zip(masks,scores):
                    mask=(mm>.5).astype(np.uint8)
                    if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
                    if int(mask.sum())<MIN_NEW_AREA:continue
                    ys,xs=np.where(mask)
                    touches=bool((x0>0 and xs.min()<6) or (x0+side<W and xs.max()>side-7)
                                 or (y0>0 and ys.min()<6) or (y0+side<H and ys.max()>side-7))
                    full=np.zeros((H,W),np.uint8);full[y0:y0+side,x0:x0+side]=mask
                    rle=enc(full)
                    raw.append({'counts':txt(rle),'score':float(score),'area':int(coco.area(rle)),
                        'source':source_name,'tile_truncated':touches,'window':[x0,y0,side]})
    raw.sort(key=lambda z:-z['score'])
    # Overlap-based grouping: merge only for strong relative overlap (min-area ratio) or IoU.
    groups=[]
    for p in raw:
        rr=as_rle(p['counts']);merged=False
        for g in groups:
            original=as_rle(g['counts'])
            inter=float(coco.area(coco.merge([rr,original],intersect=True)))
            amin=max(1.,min(float(coco.area(rr)),float(coco.area(original))))
            iou=rle_best_pairwise_iou(rr,[original])
            if iou>=STITCH_IOU or inter/amin>=STITCH_OVERLAP_FRAC:
                # Avoid joining two separately predicted filaments if their union changes
                # the mask much more than the most confident member.
                merged_rle=coco.merge([original,rr])
                g['counts']=txt(merged_rle);g['area']=int(coco.area(merged_rle))
                g['score']=max(g['score'],p['score']);g['n_support']+=1
                g['tile_truncated']=g['tile_truncated'] and p['tile_truncated']
                g['sources']=sorted(set(g['sources']+[p['source']]))
                merged=True;break
        if not merged:
            groups.append({'counts':p['counts'],'score':p['score'],'area':p['area'],
                  'tile_truncated':p['tile_truncated'],'n_support':1,'sources':[p['source']]})
        # Build a broad intermediate bank; final rank retains top candidates.
        if len(groups)>=MAX_NEW_PER_SCENE*3:break
    groups.sort(key=lambda g:(-g['score'],-g['n_support']))
    kept=groups[:MAX_NEW_PER_SCENE]
    path.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(path,'wt',compresslevel=3) as f:
        json.dump({'signature':signature,'items':kept,'raw_count':len(raw)},f,separators=(',',':'))
    return kept

def append_novel(base,novel,free_frac=FUSION_MIN_FREE,nms=FUSION_NMS_IOU):
    masks=list(base);occupied=np.zeros((H,W),np.uint8)
    for r in base:occupied|=coco.decode(r).astype(np.uint8)
    for rr in novel:
        if masks and rle_best_pairwise_iou(rr,masks)>=nms:continue
        pixel=coco.decode(rr).astype(np.uint8)
        free=pixel&(occupied==0)
        if int(free.sum())<MIN_NEW_AREA or free.sum()/max(1,pixel.sum())<free_frac:continue
        r=enc(free);masks.append(r);occupied|=free
    return masks
print('Trained V28 sources:',[x[0] for x in V28_SOURCES])

## 9. Learn which V28 detections are **truly new** vs false positives

A proposed instance is a useful positive only if it matches an official expert filament at IoU >0.5 **and that expert filament was not already matched by V25**. This is materially stricter than generic quality scoring. Acquisition-day GroupKFold supplies out-of-fold probabilities; gating thresholds are optimized for **pooled PQ** on calibration observations, not classification accuracy. The 51-scene audit is used only for final method comparison.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
NOVEL_FEATURES=['score','log_area','log_width','log_height','elongation','mask_fill',
    'source_count','support','tile_truncated','darkness','inside_variance','ring_contrast',
    'v25_max_iou','v25_nearest_dist','global_iou','tile_iou','sun_radius']

def novelty_features(stem,proposals,base):
    if not proposals:return pd.DataFrame(columns=NOVEL_FEATURES)
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    rglobal=[as_rle(p['counts']) for p in raw[SRC_HI]]
    rtile=[as_rle(p['counts']) for p in raw[SRC_TILES]]
    boxes=np.asarray([coco.toBbox(r) for r in base],float).reshape(-1,4)
    centroids=boxes[:,:2]+boxes[:,2:]/2 if len(boxes) else np.zeros((0,2))
    rows=[]
    for p in proposals:
        rr=as_rle(p['counts']);x,y,w,h=map(float,coco.toBbox(rr))
        x0=max(0,int(x)-9);y0=max(0,int(y)-9)
        x1=min(W,int(np.ceil(x+w))+9);y1=min(H,int(np.ceil(y+h))+9)
        local=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8);im=image[y0:y1,x0:x1]
        inside=im[local>0].astype(float)
        ring=(cv2.dilate(local,np.ones((7,7),np.uint8))>0)&(local==0)
        outside=im[ring].astype(float)
        if not len(inside):inside=np.asarray([0.],float)
        if not len(outside):outside=inside
        cx=x+w/2;cy=y+h/2
        mindist=(float(np.linalg.norm(centroids-np.asarray([cx,cy]),axis=1).min())/2048
                 if len(centroids) else 1.)
        rows.append(dict(score=p['score'],log_area=np.log1p(p['area']),
              log_width=np.log1p(w),log_height=np.log1p(h),
              elongation=max((w+1)/(h+1),(h+1)/(w+1)),mask_fill=p['area']/max(1,w*h),
              source_count=len(p['sources']),support=p['n_support'],
              tile_truncated=float(p['tile_truncated']),darkness=(float(outside.mean())-float(inside.mean()))/255,
              inside_variance=float(inside.std())/255,ring_contrast=float(np.median(outside)-np.median(inside))/255,
              v25_max_iou=rle_best_pairwise_iou(rr,base),v25_nearest_dist=mindist,
              global_iou=rle_best_pairwise_iou(rr,rglobal),tile_iou=rle_best_pairwise_iou(rr,rtile),
              sun_radius=float(np.hypot(cx-W/2,cy-H/2)/(W/2))))
    return pd.DataFrame(rows,columns=NOVEL_FEATURES).replace([np.inf,-np.inf],0).fillna(0.)

def matched_expert_indices(truth,pred):
    if not truth or not pred:return set()
    ious=np.asarray(coco.iou(truth,pred,[0]*len(pred)),float)
    ii,jj=linear_sum_assignment(-ious)
    return {int(a) for a,b in zip(ii,jj) if ious[a,b]>.5}

def truly_novel_labels(stem,props,base):
    labels=[]
    for p in props:
        rr=as_rle(p['counts']);votes=[]
        for ann in gt(stem):
            matched=matched_expert_indices(ann,base)
            if not ann:votes.append(False);continue
            ious=np.asarray(coco.iou([rr],ann,[0]*len(ann)),float).reshape(-1)
            best=int(ious.argmax())
            votes.append(bool(ious[best]>.5 and best not in matched))
        labels.append(int(np.mean(votes)>=.5) if votes else 0)
    return labels

# Cache candidate and feature matrices using exact V28 model checkpoint hashes.
GATE_SIG=hashlib.sha256((''.join(sha(w) for _,_,w,_,_,_ in V28_SOURCES)+sha(V25_QUALITY_PATH)).encode()).hexdigest()
def scene_novelty(stem):
    path=OUT/'gate_cache'/(stem+'.joblib')
    if path.exists():
        try:
            item=joblib.load(path)
            if item.get('signature')==GATE_SIG:return item
        except Exception:pass
    base=frozen_v25_masks(stem)
    proposed=new_detector_scene(stem)
    features=novelty_features(stem,proposed,base)
    obj={'signature':GATE_SIG,'base':[txt(r) for r in base],
         'proposed':proposed,'features':features,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def new_quality_ranker():
    return HistGradientBoostingClassifier(max_iter=125,learning_rate=.043,
        max_leaf_nodes=9,min_samples_leaf=14,l2_regularization=6.,random_state=SEED+28)

calX=[];calY=[];calG=[];calLayout={};index=0
for stem in tqdm(CAL,desc='Build V28 novel-instance calibration'):
    item=scene_novelty(stem);base=[as_rle(t) for t in item['base']]
    yy=truly_novel_labels(stem,item['proposed'],base)
    assert len(yy)==len(item['features'])
    calLayout[stem]=(index,index+len(yy));index+=len(yy)
    calX.append(item['features']);calY.extend(yy);calG.extend([stem[:8]]*len(yy))
QX=pd.concat(calX,ignore_index=True)[NOVEL_FEATURES].fillna(0.)
QY=np.asarray(calY,int);QGROUPS=np.asarray(calG)
print('Cal novel candidates',len(QY),'uniquely useful',int(QY.sum()),'positive fraction',round(QY.mean(),3) if len(QY) else 0)
GATE_ACTIVE=bool(len(QY)>=90 and QY.sum()>=12 and len(QY)-QY.sum()>=30 and len(set(QGROUPS))>=4)
GATE_PATH=Path('/kaggle/working/SOLAR_V28_NOVEL_QUALITY.joblib')
GATE_THRESH=1.01;QMODEL=None;CAL_OOF_SCORES={}
if GATE_ACTIVE:
    oof=np.full(len(QY),np.nan)
    for a,b in GroupKFold(n_splits=4).split(QX,QY,QGROUPS):
        if len(np.unique(QY[a]))<2:GATE_ACTIVE=False;break
        fold=new_quality_ranker().fit(QX.iloc[a],QY[a]);oof[b]=fold.predict_proba(QX.iloc[b])[:,1]
if GATE_ACTIVE:
    base_cal=np.zeros(4,float)
    for stem in CAL:base_cal+=physical_counts(stem,[as_rle(q) for q in scene_novelty(stem)['base']])
    base_cal_pq=pq(base_cal)
    for cut in QUALITY_THRESHOLDS:
        counts=np.zeros(4,float)
        for stem,(a,b) in calLayout.items():
            obj=scene_novelty(stem)
            old=[as_rle(z) for z in obj['base']]
            extra=[as_rle(p['counts']) for p,s in zip(obj['proposed'],oof[a:b]) if s>=cut]
            counts+=physical_counts(stem,append_novel(old,extra))
        CAL_OOF_SCORES[cut]=pq(counts)
    GATE_THRESH=max(CAL_OOF_SCORES,key=lambda t:(CAL_OOF_SCORES[t],t))
    if CAL_OOF_SCORES[GATE_THRESH]<base_cal_pq+.0005:GATE_ACTIVE=False
    else:
        QMODEL=new_quality_ranker().fit(QX,QY)
        joblib.dump({'model':QMODEL,'features':NOVEL_FEATURES,'threshold':GATE_THRESH,
                     'oof_auc':roc_auc_score(QY,oof),'train_calibration_only':True},GATE_PATH)
if not GATE_ACTIVE:
    QMODEL=None;GATE_THRESH=1.01
    print('Novel instance gate DISABLED: insufficient unique positives or no CAL out-of-fold PQ gain.')
else:print('Novel detection quality gate enabled, threshold',GATE_THRESH,'CAL OOF PQ',CAL_OOF_SCORES)
save_json(OUT/'reports'/'V28_quality_report.json',{'active':GATE_ACTIVE,'examples':len(QY),
    'unique_positives':int(QY.sum()),'threshold':GATE_THRESH,'oof_cal_PQ':CAL_OOF_SCORES,
    'calibration_scenes':len(CAL),'audit_scenes':len(AUDIT),'labels_from_official_train_only':True})

## 10. Independent-day PQ ablations with protected 0.39 control

Evaluate the new high-resolution detector **alone**, raw confidence fusion, conservative confidence fusion and trained novel-quality fusion. These are different hypotheses, not equivalent threshold copies. An optional learned fusion is only enabled if it improved calibration out-of-fold PQ. Method promotion requires audit + paired-bootstrap evidence; otherwise `submission.csv` contains the exact historical V25 file.

In [ ]:
V28_METHODS=['V25_PROVEN_039','V28_HR_ALONE','V28_RAW_CONF_FUSION',
             'V28_CONSERVATIVE_FUSION','V28_LEARNED_NOVEL_FUSION','V28_STRICT_NOVEL_FUSION']

def method_masks(stem,name,item=None):
    obj=item if item is not None else scene_novelty(stem)
    base=[as_rle(q) for q in obj['base']]
    props=obj['proposed']
    if name=='V25_PROVEN_039':return base
    if name=='V28_HR_ALONE':
        filtered=[as_rle(p['counts']) for p in props if p['score']>=.46]
        return append_novel([],filtered,free_frac=.75,nms=.32)
    if name=='V28_RAW_CONF_FUSION':
        filtered=[as_rle(p['counts']) for p in props if p['score']>=.55]
        return append_novel(base,filtered)
    if name=='V28_CONSERVATIVE_FUSION':
        filtered=[as_rle(p['counts']) for p in props if p['score']>=.75 and p['n_support']>=2]
        return append_novel(base,filtered,free_frac=.82,nms=.18)
    if name in ('V28_LEARNED_NOVEL_FUSION','V28_STRICT_NOVEL_FUSION'):
        if not GATE_ACTIVE:return base
        probs=QMODEL.predict_proba(obj['features'][NOVEL_FEATURES].fillna(0.))[:,1] if len(props) else []
        threshold=GATE_THRESH+(.12 if name=='V28_STRICT_NOVEL_FUSION' else 0)
        filtered=[as_rle(p['counts']) for p,z in zip(props,probs) if z>=threshold]
        return append_novel(base,filtered,free_frac=.80 if name=='V28_STRICT_NOVEL_FUSION' else FUSION_MIN_FREE,
                            nms=.18 if name=='V28_STRICT_NOVEL_FUSION' else FUSION_NMS_IOU)
    raise ValueError(name)

AUDIT_SUM={name:np.zeros(4,float) for name in V28_METHODS}
AUDIT_RECORD=[]
for stem in tqdm(AUDIT,desc='V28 held-day audit'):
    obj=scene_novelty(stem)
    for name in V28_METHODS:
        masks=method_masks(stem,name,obj)
        c=physical_counts(stem,masks);AUDIT_SUM[name]+=c
        AUDIT_RECORD.append({'stem':stem,'day':stem[:8],'method':name,'sum_iou':float(c[0]),
           'TP':c[1],'FP':c[2],'FN':c[3],'instances':len(masks)})
CONTROL='V25_PROVEN_039'
BASE_AUDIT_PQ=pq(AUDIT_SUM[CONTROL])
report=pd.DataFrame([{'method':name,**metrics(c),'delta_vs_v25':pq(c)-BASE_AUDIT_PQ}
       for name,c in AUDIT_SUM.items()]).sort_values('PQ',ascending=False)
report.to_csv(OUT/'reports'/'V28_AUDIT_PQ.csv',index=False)
pd.DataFrame(AUDIT_RECORD).to_csv(OUT/'reports'/'V28_AUDIT_PER_SCENE.csv',index=False)
display(report)
assert abs(BASE_AUDIT_PQ-.458880)<.032,'V25 frozen audit no longer matches previous experiment; inspect before promoting model.'

perday=pd.DataFrame(AUDIT_RECORD).groupby(['method','day'])[['sum_iou','TP','FP','FN']].sum()
days=sorted({r['day'] for r in AUDIT_RECORD})
arr={k:np.stack([perday.loc[(k,d)].to_numpy(float) for d in days]) for k in V28_METHODS}
ix=np.random.default_rng(SEED+28).integers(0,len(days),size=(1500,len(days)))
def vector_pq(z):
    denom=z[:,1]+.5*(z[:,2]+z[:,3]);return np.divide(z[:,0],denom,out=np.zeros(len(z)),where=denom>0)
control_boot=vector_pq(arr[CONTROL][ix].sum(axis=1))
bootstrap=[]
for k in V28_METHODS:
    if k==CONTROL:continue
    dif=vector_pq(arr[k][ix].sum(axis=1))-control_boot
    bootstrap.append({'method':k,'delta_audit_PQ':pq(AUDIT_SUM[k])-BASE_AUDIT_PQ,
        'delta_ci_2p5':float(np.quantile(dif,.025)),
        'delta_ci_97p5':float(np.quantile(dif,.975)),
        'bootstrap_fraction_positive':float(np.mean(dif>0))})
bootdf=pd.DataFrame(bootstrap).sort_values('delta_audit_PQ',ascending=False)
bootdf.to_csv(OUT/'reports'/'V28_PAIRED_DATE_BOOTSTRAP.csv',index=False)
display(bootdf)
eligible=bootdf[(bootdf.delta_audit_PQ>=MIN_LOCAL_GAIN_TO_PROMOTE)&(bootdf.bootstrap_fraction_positive>=.90)]
SELECTED=str(eligible.iloc[0].method) if len(eligible) else CONTROL
print('Conservative candidate selected:',SELECTED,'| new public score remains unverified')
save_json(OUT/'reports'/'V28_selection.json',{'default':SELECTED,'known_public_v25':.39,
   'new_public_score':None,'audit_PQ':{k:pq(v) for k,v in AUDIT_SUM.items()},
   'historical_validation_reuse':True,'learned_gate_active':GATE_ACTIVE,
   'test_labels_used':False,'trained_hr_checkpoint_sha':sha(HR_ALIAS)})

## 11. Write test submissions; enforce the **exact** known-public CSV SHA

New predictions are generated for all 180 test images and exported with their method names. Historical V25 is recovered byte-for-byte (`ac470fbf...` SHA prefix); regenerated CSVs with different hashes are never presented as the already-scored result. Test images are only used for inference, not training or model selection.

In [ ]:
import csv
TEST_STEMS=sorted(TEST);assert len(TEST_STEMS)==180
PROVEN=Path('/kaggle/working/V28_PROTECTED_V25_PUBLIC_039.csv')
verified=False
for name in ('V25_REAL_REFINE_PLUS_QUALITY.csv','submission.csv'):
    data,origin=find_original_asset(name)
    if data is not None and hashlib.sha256(data).hexdigest().startswith(V25_EXPECTED_CSV_SHA):
        PROVEN.write_bytes(data);verified=True
        print('Recovered exact V25 0.39 CSV from:',origin);break

if ALWAYS_EXPORT_ABLATIONS:
    for stem in tqdm(TEST_STEMS,desc='V28 test inference + resumable caches'):
        scene_novelty(stem)

CSV_PATHS={name:OUT/'submissions'/(name+'.csv') for name in V28_METHODS}
count_rows=[]
from contextlib import ExitStack
with ExitStack() as stack:
    writers={name:csv.writer(stack.enter_context(open(path,'w',encoding='utf8',newline='')))
             for name,path in CSV_PATHS.items()}
    for wr in writers.values():wr.writerow(['filament_id','segmentation_rle'])
    for stem in tqdm(TEST_STEMS,desc='Write per-branch experiment CSVs'):
        obj=scene_novelty(stem)
        for name,wr in writers.items():
            masks=method_masks(stem,name,obj)
            count_rows.append({'stem':stem,'method':name,'instances':len(masks)})
            for idx,rr in enumerate(masks,1):wr.writerow([f'{stem}_{idx}',txt(rr)])

count_df=pd.DataFrame(count_rows);count_df.to_csv(OUT/'reports'/'V28_TEST_COUNTS.csv',index=False)
manifest={}
for name,path in CSV_PATHS.items():
    frame=pd.read_csv(path,dtype=str)
    assert frame.columns.tolist()==['filament_id','segmentation_rle'] and frame.filament_id.is_unique
    assert frame.segmentation_rle.notna().all()
    assert frame.filament_id.str.rsplit('_',n=1).str[0].isin(TEST).all()
    assert int(count_df[count_df.method==name].instances.sum())==len(frame)
    for v in frame.segmentation_rle:assert int(coco.area(as_rle(v)))>0
    manifest[name]={'rows':len(frame),'sha256':sha(path),'file':str(path)}
    print(name,'rows',len(frame),'SHA',sha(path)[:18])

# Historical regenerate may differ due serialization or mask boundary tie ordering.
if not verified and sha(CSV_PATHS[CONTROL]).startswith(V25_EXPECTED_CSV_SHA):
    shutil.copy2(CSV_PATHS[CONTROL],PROVEN);verified=True
assert verified and PROVEN.exists() and sha(PROVEN).startswith(V25_EXPECTED_CSV_SHA),(
  'Exact historical V25 submission not attached. Please attach V25 predictions/reports ZIP. '
  'This notebook does not mislabel an unverified regenerated CSV as your confirmed 0.39 winner.')
assert len(pd.read_csv(PROVEN))==V25_EXPECTED_ROWS
SUBMISSION=Path('/kaggle/working/submission.csv')
shutil.copy2(PROVEN if SELECTED==CONTROL else CSV_PATHS[SELECTED],SUBMISSION)
for i,name in enumerate([s for s in V28_METHODS if s!=CONTROL],1):
    shutil.copy2(CSV_PATHS[name],Path('/kaggle/working')/f'V28_ALT_{i}.csv')
save_json(OUT/'reports'/'V28_manifest.json',{'selected':SELECTED,
   'confirmed_historical_V25_SHA':sha(PROVEN),'default_submission_SHA':sha(SUBMISSION),
   'new_model_public_score':None,'test_label_access':False,'candidates':manifest,
   'highres_model_sha':sha(HR_ALIAS),'quality_gate_active':GATE_ACTIVE})
print('DEFAULT submission.csv:',SELECTED,'file',SUBMISSION)

## 12. Save full model and reports archives

Use **Save Version → Save & Run All** with Kaggle outputs. `SOLAR_V28_HIGHRES_BEST.pt` is saved **before** expensive test inference. `SOLAR_V28_ALL_TRAINED_CHECKPOINTS.zip` includes the new and frozen old weights, quality model (when trained), CSVs and run manifest. Save this archive as a Kaggle Dataset to avoid retraining on subsequent runs.

In [ ]:
REPORT_ZIP=Path('/kaggle/working/SOLAR_V28_PREDICTIONS_AND_REPORTS.zip')
with zipfile.ZipFile(REPORT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
    for p in sorted((OUT/'reports').glob('V28_*.csv'))+sorted((OUT/'reports').glob('V28_*.json')):
        z.write(p,arcname='reports/'+p.name)
    for p in sorted((OUT/'submissions').glob('*.csv')):z.write(p,arcname='submissions/'+p.name)
    z.write(SUBMISSION,arcname='submission.csv')
    z.write(PROVEN,arcname=PROVEN.name)
with zipfile.ZipFile(REPORT_ZIP) as z:assert z.testzip() is None
MODEL_ZIP=Path('/kaggle/working/SOLAR_V28_ALL_TRAINED_CHECKPOINTS.zip')
with zipfile.ZipFile(MODEL_ZIP,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for p in WEIGHTS.values():z.write(p,arcname='detectors/'+p.name)
    for p in (V24_REFINER_PATH,V24_QUALITY_PATH,V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH,HR_ALIAS,CTX_ALIAS,GATE_PATH):
        if p is not None and Path(p).is_file():z.write(p,arcname='models/'+Path(p).name)
    z.write(HR_ROOT/'dataset_manifest.json',arcname='reports/hr_dataset_manifest.json')
    z.write(REPORT_ZIP,arcname=REPORT_ZIP.name)
with zipfile.ZipFile(MODEL_ZIP) as z:assert z.testzip() is None
print('ALL FINISHED. Verified previous public PQ 0.39 retained; new V28 public score unverified.')
print('Submission:',SUBMISSION)
print('Reports:',REPORT_ZIP,'MiB',round(REPORT_ZIP.stat().st_size/2**20,2))
print('New/old trained models:',MODEL_ZIP,'MiB',round(MODEL_ZIP.stat().st_size/2**20,2))

### Read results before submitting

Check `solar_v28/reports/V28_AUDIT_PQ.csv`, `V28_PAIRED_DATE_BOOTSTRAP.csv` and `V28_selection.json`. The chosen `submission.csv` is either **byte-identical to your previously scored V25 0.39** or comes from a method that cleared the audit gate. The five `V28_ALT_*.csv` files let you inspect separately which new approach was tested. Do not submit the control again; a .40 claim requires an actual new Kaggle result.

**Key research caveats:** new detector fine-tuning begins with V20 weights trained on the 603 historical training scenes. The 104-scene comparison set has been reused extensively in previous versions. YOLO polygons simplify disconnected annotation islands (conversion records the loss fraction), and overlapping crops may still fragment very long filaments. No external labels, exposed test masks or scoring loopholes are used.